In [2]:
def approx_vertex_cover(graph):
    edges = set()
    for u in graph:
        for v in graph[u]: # Corrected 'grap' to 'graph'
            edge = tuple(sorted((u,v)))
            edges.add(edge)

    cover = set()

    while edges:
        u, v = edges.pop()
        cover.add(u)
        cover.add(v)
        edges = {e for e in edges if u not in e and v not in e}
    return cover



if __name__ == "__main__":
  grafo = {
      1: {2},
      2: {1,3},
      3: {2,4},
      4: {3,5},
      5: {4,6},
      6: {5,7},
      7: {6,8},
      8: {7}
  }


resultado = approx_vertex_cover(grafo)
print("vertex cover aproximado: ", resultado)
print("tamaño:", len(resultado))

vertex cover aproximado:  {1, 2, 3, 4, 6, 7}
tamaño: 6


In [3]:
def es_vertex_cover_valido(graph, cover):
    for u in graph:
        for v in graph[u]:
            if u not in cover and v not in cover:
                return False

    return True



grafo = {
    1: {2},
    2: {1,3},
    3: {2,4},
    4: {3,5},
    5: {4,6},
    6: {5,7},
    7: {6,8},
    8: {7}
}


cover = approx_vertex_cover(grafo)
print("cover:", cover)
print("es valido?", es_vertex_cover_valido(grafo, cover))



cover: {1, 2, 3, 4, 6, 7}
es valido? True


In [4]:
def approx_vertex_cover_eficiente(graph):
    covered = set()
    cover = set()


    edges = []
    seen = set()
    for u in graph:
        for v in graph[u]:
            e = tuple(sorted((u,v)))
            if e not in seen:
                seen.add(e)
                edges.append(e)

    for u,v in edges:
        if u not in covered and v not in covered:
            cover.add(u)
            cover.add(v)
            covered.add(u)
            covered.add(v)

    return cover



In [7]:
import numpy as np


def gonzalez_k_center(points, k, seed= 0):

    points = np.array(points)
    n = len(points)
    rng = np.random.default_rng(seed)
    center = [rng.integers(n)]

    dist_to_nearest = np.linalg.norm(points - points[center[0]], axis = 1)

    while len(center) < k:
        next_center = np.argmax(dist_to_nearest)
        center.append(next_center)

        new_dist = np.linalg.norm(points - points[next_center], axis=1)
        dist_to_nearest = np.minimum(dist_to_nearest, new_dist)

    centers_coords = points[center]
    dists = np.linalg.norm(points[:,None, :] - centers_coords[None, : , : ], axis = 2)
    assignment = np.argmin(dists, axis = 1)
    radius = dists[np.arange(n), assignment].max()

    return center, assignment, radius



if __name__ == "__main__":
    np.random.seed(42)
    puntos = np.random.rand(20,2) * 10

    centros_idx, asignacion, radio = gonzalez_k_center(puntos, k = 3)

    print("Indices de centros: ", centros_idx)
    print("Asignacion de cada punto: ", asignacion)
    print("Radio maximo (cota < 2 opt):", radio)

Indices de centros:  [np.int64(17), np.int64(2), np.int64(5)]
Asignacion de cada punto:  [2 0 1 2 0 2 0 1 1 1 1 1 2 1 1 1 2 0 1 0]
Radio maximo (cota < 2 opt): 6.107575923656371


In [10]:
import itertools
import networkx as nx

def mst_approx_tsp(G):
    T = nx.minimum_spanning_tree(G, weight='weight')

    H = nx.MultiGraph()
    H.add_nodes_from(T.nodes())
    for u, v, w in T.edges(data='weight'):
        H.add_edge(u,v, weight=w)
        H.add_edge(u,v, weight=w)

    circuito = list(nx.eulerian_circuit(H, source = list(G.nodes())[0]))

    visitados = set()
    tour = []
    for u, v in circuito:
        if u not in visitados:
            tour.append(u)
            visitados.add(u)
    tour.append(tour[0])

    return tour


def christofides_tsp(G):
    tour = nx.approximation.christofides(G, weight = 'weight')
    return tour

def costo_tour(G, tour):
    return sum(G[tour[i]][tour[i+1]]['weight'] for i in range(len(tour) - 1))



if __name__ == "__main__":
    import math
    import random

    random.seed(0)
    n = 8
    puntos = {i: (random.uniform(0,100), random.uniform(0,100)) for i in range(n)}

    G = nx.complete_graph(n)
    for u, v in G.edges():
        x1, y1 = puntos[u]
        x2,y2 = puntos[v]
        G[u][v]['weight'] = math.dist((x1,y1), (x2,y2))

    tour_mst = mst_approx_tsp(G)
    tour_chris = christofides_tsp(G)

    print("tour 2 aprox", tour_mst)
    print("cost", costo_tour(G, tour_mst))


    print("tour ", tour_chris)
    print("costo", costo_tour(G, tour_chris))

tour 2 aprox [0, 5, 3, 7, 2, 4, 6, 1, 0]
cost 247.19265549803615
tour  [0, 6, 4, 2, 1, 7, 3, 5, 0]
costo 204.58239476035675


In [12]:
def knapsack_greedy(items, capacity):
    items_ordenados = sorted(items, key=lambda  x: x[0] / x[1], reverse=True)
    peso_actual = 0
    valor_actual = 0
    elegidos = []

    for valor, peso, nombre in items_ordenados:
        if peso_actual + peso <= capacity:
            elegidos.append((valor, peso, nombre))
            peso_actual += peso
            valor_actual += valor

    items_que_caben = [it for it in items if it[1] <= capacity]
    if items_que_caben:
        mejor_individual = max(items_que_caben, key = lambda x: x[0])
    else:
        mejor_individual = None

    if mejor_individual and mejor_individual[0] > valor_actual:
        return mejor_individual[0], [mejor_individual]

    return valor_actual, elegidos


if __name__ == "__main__":
    items = [
        (60, 10, "A"),
        (100, 20, "B"),
        (120, 30, "C")
    ]
    capacidad = 50

    valor, elegidos = knapsack_greedy(items, capacidad)
    print("valor aproximaco:", valor)
    print("items elegidos: ", [nombre for _, _, nombre in elegidos])

valor aproximaco: 160
items elegidos:  ['A', 'B']
